# Bayesian fault-geometry inference with the collapsed sampler

This example runs full posterior inference for planar-fault geometry with
`geodef.bayes`: a **collapsed** (Rao-Blackwellized) formulation in which the
linear slip parameters are marginalized analytically and NUTS samples only the
geometry and scale hyperparameters. See `docs/bayes.md` for the theory.

We build a synthetic scenario (the tutorial 10/11 setup), then:

1. find the MAP geometry with `geometry_search` (Gauss-Newton error bars),
2. sample the full posterior with NUTS (`mode='hierarchical'`),
3. compare the two — where the Laplace approximation is enough and where it is not,
4. propagate geometry uncertainty into **slip credible intervals**,
5. compare against the `'weak'` (effectively unsmoothed) prior mode, and
6. cross-check the posterior against gradient-free `emcee` on the same density.

Requires `pip install geodef[bayes]` (JAX + blackjax); the emcee section also
needs `pip install emcee`.

## Question, provenance, and execution contract

**Question.** How do full nonlinear geometry posteriors differ from local
Gauss–Newton uncertainty, and how much do slip intervals depend on the prior?

**Data provenance.** Every observation in this notebook is generated below
from a declared planar fault. `np.random.default_rng(0)` supplies the only
random input; no files or downloads are required.

**Environment.** Requires `geodef[bayes]` (JAX and BlackJAX). Sampling uses
explicit seeds. The full notebook is an advanced worked example; the tutorial
suite uses a smaller Bayesian chapter for routine execution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import geodef

geodef.backend.set_backend("jax")

## 1. Synthetic scenario

A buried thrust fault (dip 15, depth 25 km) discretized 4x3, slipping in a
smooth dip-slip bump, observed by a 6x6 GNSS network with 2 mm noise.

In [ ]:
REF_LAT, REF_LON = -2.0, 100.0
TRUE = dict(depth=25e3, strike=315.0, dip=15.0, length=120e3, width=60e3)
NL, NW = 4, 3

fault_true = geodef.Fault.planar(
    lat=REF_LAT, lon=REF_LON, n_length=NL, n_width=NW, **TRUE
)
i = np.arange(fault_true.n_patches) % NL
j = np.arange(fault_true.n_patches) // NL
slip_true = 3.0 * np.exp(-(((i - 1.5) / 1.5) ** 2 + (j - 1.0) ** 2))

glon, glat = np.meshgrid(np.linspace(99.0, 101.0, 6), np.linspace(-3.0, -1.0, 6))
glon, glat = glon.ravel(), glat.ravel()
ue, un, uz = fault_true.displacement(
    glat, glon, np.zeros_like(slip_true), slip_true
)

rng = np.random.default_rng(42)
sigma = 0.002
n = len(glat)
gnss = geodef.GNSS(
    lon=glon, lat=glat,
    ve=ue + rng.normal(0, sigma, n),
    vn=un + rng.normal(0, sigma, n),
    vu=uz + rng.normal(0, sigma, n),
    se=np.full(n, sigma), sn=np.full(n, sigma), su=np.full(n, sigma),
)

theta_true = np.array([0.0, 0.0, TRUE["depth"], TRUE["strike"],
                       TRUE["dip"], TRUE["length"], TRUE["width"]])

## 2. MAP geometry with Gauss-Newton error bars

`geometry_search` gives the maximum a posteriori point (for flat priors) and a
local Gaussian approximation of uncertainty — the Laplace picture the sampler
will either confirm or overturn.

In [ ]:
theta0 = theta_true.copy()
theta0[4] = 25.0    # start dip well away from the truth
theta0[2] = 35e3    # and depth too

search = geodef.invert.geometry_search(
    theta0, gnss, ref_lat=REF_LAT, ref_lon=REF_LON,
    free=["dip", "depth"],
    bounds={"dip": (5.0, 45.0), "depth": (10e3, 60e3)},
    n_length=NL, n_width=NW, components="dip",
    regularization="laplacian", regularization_strength=1.0,
)
gn_sd = np.sqrt(np.diag(search.theta_cov))
print(f"MAP dip   = {search.theta[4]:8.3f} +/- {gn_sd[0]:.3f} deg (true {TRUE['dip']})")
print(f"MAP depth = {search.theta[2]/1e3:8.3f} +/- {gn_sd[1]/1e3:.3f} km (true {TRUE['depth']/1e3})")

## 3. Full posterior with NUTS

The hierarchical mode also samples `log10_sigma` (noise scale) and
`log10_lambda` (smoothing strength), so the results average over all
regularization strengths weighted by the evidence — nothing is tuned by
hand. We start the sampler at the MAP geometry.

In [ ]:
post = geodef.bayes.RectPosterior(
    search.theta, gnss, ref_lat=REF_LAT, ref_lon=REF_LON,
    free=["dip", "depth"],
    theta_prior={"dip": (5.0, 45.0), "depth": (10e3, 60e3)},
    n_length=NL, n_width=NW, components="dip",
    mode="hierarchical", regularization="laplacian", regularization_strength=1.0,
)

result = geodef.bayes.sample(
    post, n_samples=1500, n_warmup=1000, n_chains=4, seed=0
)

stats = result.summary()
print(f"{'param':>14s} {'mean':>10s} {'sd':>9s} {'rhat':>6s} {'ess':>7s}")
for k, name in enumerate(result.param_names):
    print(f"{name:>14s} {stats['mean'][k]:10.3f} {stats['sd'][k]:9.3f} "
          f"{stats['rhat'][k]:6.3f} {stats['ess'][k]:7.0f}")
print(f"divergences: {result.n_divergent}, acceptance: {result.acceptance_rate:.2f}")

In [ ]:
fig, axes = result.plot_pairs()
fig.suptitle("Joint posterior: geometry + scales", y=1.02);

## 4. Laplace vs full posterior

For a well-resolved, nearly linear problem the two agree; the full posterior
earns its keep when the geometry trades off against depth (banana-shaped
contours in the pair plot), when bounds truncate the posterior, or when
`log10_lambda` is poorly constrained.

In [ ]:
dip_s, depth_s = result.flat[:, 0], result.flat[:, 1]
print(f"{'':>10s} {'Gauss-Newton':>16s} {'posterior':>16s}")
print(f"{'dip':>10s} {search.theta[4]:8.3f} +/- {gn_sd[0]:5.3f} "
      f"{dip_s.mean():8.3f} +/- {dip_s.std():5.3f}")
print(f"{'depth km':>10s} {search.theta[2]/1e3:8.3f} +/- {gn_sd[1]/1e3:5.3f} "
      f"{depth_s.mean()/1e3:8.3f} +/- {depth_s.std()/1e3:5.3f}")
corr = np.corrcoef(dip_s, depth_s)[0, 1]
print(f"posterior dip-depth correlation: {corr:+.2f}")

## 5. Slip credible intervals that include geometry uncertainty

One exact Gaussian conditional draw per posterior sample gives draws from the
joint posterior of slip *and* geometry — per-patch intervals no fixed-geometry
inversion can produce.

In [ ]:
thin = result.flat[::10]
draws = post.slip_draws(thin, seed=1)          # (n, N) dip-slip columns
slip_mean = draws.mean(axis=0)
slip_lo, slip_hi = np.percentile(draws, [5, 95], axis=0)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), constrained_layout=True)
for ax, field, title in [
    (axes[0], slip_mean, "posterior mean slip (m)"),
    (axes[1], slip_true, "true slip (m)"),
    (axes[2], slip_hi - slip_lo, "90% credible width (m)"),
]:
    im = ax.imshow(field.reshape(NW, NL), origin="upper", cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("along strike"); ax.set_ylabel("down dip")
    fig.colorbar(im, ax=ax, shrink=0.8)

## 6. The 'weak' prior mode — the collapsed analog of unsmoothed MCMC

Replace the Laplacian + sampled lambda with a fixed, wide independent
Gaussian prior on each patch (`slip_scale` meters). Resolved patches stay
tight; poorly resolved (deep) patches show honestly wide intervals — the
usual selling point of joint 'unsmoothed' samplers, at collapsed-sampler
cost.

In [ ]:
post_weak = geodef.bayes.RectPosterior(
    search.theta, gnss, ref_lat=REF_LAT, ref_lon=REF_LON,
    free=["dip", "depth"],
    theta_prior={"dip": (5.0, 45.0), "depth": (10e3, 60e3)},
    n_length=NL, n_width=NW, components="dip",
    mode="weak", regularization=None, slip_scale=10.0,
)
result_weak = geodef.bayes.sample(
    post_weak, n_samples=1500, n_warmup=1000, n_chains=4, seed=0
)
draws_weak = post_weak.slip_draws(result_weak.flat[::10], seed=1)
w_h = np.percentile(draws_weak, 95, axis=0) - np.percentile(draws_weak, 5, axis=0)
w_s = slip_hi - slip_lo

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(w_s, "o-", label="hierarchical (Laplacian, sampled $\\lambda$)")
ax.plot(w_h, "s-", label="weak prior (slip_scale = 10 m)")
ax.set_xlabel("patch index"); ax.set_ylabel("90% credible width (m)")
ax.legend(); ax.set_title("How much did the smoothing prior matter?");

## 7. Cross-check with emcee

`emcee` explores the same `logpdf` with gradient-free ensemble moves. It needs
tens of thousands of density evaluations (vs thousands of gradient evaluations
for NUTS) but must land on the same posterior — a strong independent check.
`jax.jit` makes each call cheap.

In [ ]:
import emcee
import jax

logpdf = jax.jit(post.logpdf)
ndim = post.n_params
nwalkers = 2 * ndim + 12
p0 = result.flat.mean(axis=0) + 3 * result.flat.std(axis=0) * rng.standard_normal(
    (nwalkers, ndim)
)
sampler = emcee.EnsembleSampler(nwalkers, ndim, lambda x: float(logpdf(x)))
sampler.run_mcmc(p0, 2000, progress=False)
chain = sampler.get_chain(discard=500, flat=True)

print(f"{'param':>14s} {'NUTS mean+/-sd':>20s} {'emcee mean+/-sd':>20s}")
for k, name in enumerate(result.param_names):
    print(f"{name:>14s} {result.flat[:,k].mean():10.3f} +/- {result.flat[:,k].std():7.3f} "
          f"{chain[:,k].mean():10.3f} +/- {chain[:,k].std():7.3f}")

## Takeaways

- The collapsed posterior turns a several-hundred-dimensional Bayesian slip
  inversion into a handful of NUTS dimensions; a full run takes minutes on CPU.
- Sampling `log10_lambda` replaces picking the ABIC minimum: slip intervals
  include regularization uncertainty.
- The `'weak'` mode answers "how much did the smoothing prior matter?" within
  the same framework.
- What it cannot do: positivity-constrained slip priors (they break the
  Gaussian marginalization) — that joint sampler is future, GPU-scale work,
  as is triangular-mesh geometry (pending `tri` jit/vmap support).

## Assumptions, validation, and interpretation

The likelihood assumes the declared GNSS covariance and the elastic half-space;
geometry priors and slip-prior modes are explicit in each posterior. Compare
chains, R-hat, ESS, divergences, prior sensitivity, and predictive behavior
before interpreting parameter intervals. Agreement with the synthetic truth is
a controlled validation here, not evidence that the same model family is
adequate for real data.